# vedit · говорящий аватар на бесплатном GPU Colab

**Что делает:** текст → речь (Chatterbox Multilingual, клон голоса по образцу) → говорящее фото (JoyVASA + LivePortrait) → `avatar_result.zip` для монтажа в vedit.

**Как запустить**
1. *Runtime → Change runtime type →* **T4 GPU** → Save.
2. *Runtime → Run all* (Ctrl+F9).
3. Ячейка **«4. Загрузка»** попросит файл — выберите `avatar_job.zip` (его делает `vedit avatar export …` или бот по команде `/avatar`).
4. В конце браузер скачает `avatar_result.zip` → `vedit avatar import <job> avatar_result.zip` или пришлите архив боту документом.

Первый запуск: установка и модели ≈ 8–12 минут (≈ 9 ГБ скачивается внутри Colab, не у вас). Повторный запуск в той же сессии пропускает всё готовое. Сам ролик: ≈ 1–2 минуты речи на 30 секунд и ≈ 2–4 минуты анимации.

**Режимы** (определяются архивом, можно поменять в ячейке «Настройки»):
- `avatar` — голос + видео (обычный);
- `voice_only` — только голос (`speech.wav`), без видео;
- если в архиве уже есть готовая речь (`speech_in.wav`: ваша запись `vedit avatar export --speech my.ogg` или голос из каталога, синтезированный локально Piper, `--voice denis`) — Chatterbox **не устанавливается и не запускается**, ноутбук только анимирует лицо (быстрее на ~3–5 минут);
- голоса-клоны из каталога (`--voice chatterbox_denis`) приходят с образцом `voice_ref.wav`, Chatterbox говорит этим тембром.

**Лицензии:** Chatterbox — MIT (в синтезированной речи неслышимый водяной знак Perth от Resemble AI); JoyVASA, LivePortrait, chinese-hubert-base — MIT.
⚠️ Модели детекции лица InsightFace (buffalo_l), которые использует LivePortrait, разрешены **только для некоммерческих исследований**. Для тестов — можно; до коммерческого запуска их нужно заменить (например, на MediaPipe Face Detection).

## Почему два окружения (и почему Colab-овский torch не трогаем)

`chatterbox-tts` жёстко закрепляет `torch==2.6.0` и `transformers==5.2.0`, а JoyVASA написан под `transformers 4.39` (его аудиоэнкодер наследует внутренние классы HuBERT, в 5.x они другие) и `numpy 1.26`. В одном окружении они несовместимы, а ставить их поверх предустановленных пакетов Colab — значит сломать ядро ноутбука и ловить «перезапустите runtime».

Поэтому:
- создаются **два изолированных venv** через `uv` (быстрый установщик, уже есть в Colab): `venv_tts` (Chatterbox) и `venv_joy` (JoyVASA), оба на Python 3.11 — `uv` сам скачает его, если в Colab другая версия;
- в обоих один и тот же `torch 2.6.0` (CUDA 12.4): `uv` кэширует колёса, поэтому torch **скачивается один раз**;
- ядро Colab и его torch не меняются → **перезапуск runtime не нужен**, ячейки можно перезапускать сколько угодно;
- речь и анимация идут отдельными процессами: видеопамять T4 (15 ГБ) освобождается между этапами.

JoyVASA тестировался авторами на torch 2.2; на 2.6 нужен только флаг `TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1` (его чекпойнт хранит аргументы обучения через pickle) — скрипт ставит его сам.

In [ ]:
#@title 1. Проверка GPU
import subprocess, shutil
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit('❌ GPU не подключён: Runtime → Change runtime type → T4 GPU → Save, затем Runtime → Run all')
print('GPU:', r.stdout.strip())
free = shutil.disk_usage('/content').free / 1e9
print(f'свободно на диске: {free:.0f} ГБ')
assert free > 15, 'нужно ≥ 15 ГБ свободного места'

In [ ]:
#@title 2. Установка анимации (≈ 3–4 минуты в первый раз; повторно — мгновенно). Chatterbox ставится в ячейке 6, только если он нужен
import hashlib, os, shutil, subprocess, sys, time
ROOT = '/content/vedit_avatar'
JOY_DIR = f'{ROOT}/JoyVASA'
JOYVASA_COMMIT = '916a90f8de490e8648fee460c1200bd5d9a795af'
TTS_PY = f'{ROOT}/venv_tts/bin/python'
JOY_PY = f'{ROOT}/venv_joy/bin/python'
os.makedirs(ROOT, exist_ok=True)

REQUIREMENTS = {
    'tts': '''# Chatterbox Multilingual TTS (MIT). Pins torch 2.6.0 / transformers 5.2.0 itself — kept in its own venv.
chatterbox-tts==0.1.7
# resemble-perth (the watermarker) imports pkg_resources; uv venvs have no setuptools and setuptools>=81 drops it
setuptools<81
''',
    'joy': '''# JoyVASA (human mode) inference only — the subset of JoyVASA's requirements.txt the human
# inference path imports, at the versions the authors tested (transformers 4.39.2, numpy 1.26.4).
# torch 2.6.0 instead of 2.2.2: the same wheel as chatterbox-tts, so uv downloads it once
# (needs TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 — vedit_avatar.py sets it).
# No xformers / bitsandbytes / gradio / audio-separator / XPose: not used by human inference.
torch==2.6.0
torchvision==0.21.0
numpy==1.26.4
opencv-python-headless==4.10.0.84
librosa==0.10.2.post1
soundfile
transformers==4.39.2
huggingface_hub>=0.23,<1.0
onnxruntime==1.19.2
onnx==1.16.1
scipy==1.13.1
scikit-image==0.24.0
imageio==2.34.2
imageio-ffmpeg==0.5.1
pykalman==0.9.7
rich==13.7.1
pyyaml==6.0.1
tyro==0.8.5
matplotlib==3.9.0
pillow==10.3.0
tqdm==4.66.4
''',
}

def sh(cmd, **kw):
    """Runs a command and streams its output into the notebook."""
    print('$', cmd if isinstance(cmd, str) else ' '.join(map(str, cmd)), flush=True)
    p = subprocess.Popen(cmd, shell=isinstance(cmd, str), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, **kw)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'команда завершилась с кодом {p.returncode} (вывод выше)')

t0 = time.time()
if not shutil.which('ffmpeg'):
    sh('apt-get -qq update && apt-get -qq install -y ffmpeg')
if not shutil.which('uv'):
    sh([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])

# JoyVASA at a fixed commit (the one vedit was checked against)
if not os.path.exists(f'{JOY_DIR}/inference.py'):
    shutil.rmtree(JOY_DIR, ignore_errors=True)
    sh(f'git init -q {JOY_DIR} && cd {JOY_DIR} && git remote add origin https://github.com/jdh-algo/JoyVASA.git '
       f'&& git fetch -q --depth 1 origin {JOYVASA_COMMIT} && git checkout -q FETCH_HEAD')

def ensure_venv(name):
    """Creates venv_<name> from REQUIREMENTS[name] once (a stamp holds the requirements hash)."""
    req = REQUIREMENTS[name]
    venv = f'{ROOT}/venv_{name}'
    stamp = f'{venv}/.vedit_ok'
    want = hashlib.sha256(req.encode()).hexdigest()
    if os.path.exists(stamp) and open(stamp).read() == want:
        print(f'venv_{name}: уже установлен')
        return
    shutil.rmtree(venv, ignore_errors=True)
    req_file = f'{ROOT}/requirements-{name}.txt'
    open(req_file, 'w').write(req)
    sh(['uv', 'venv', '-q', '--python', '3.11', venv])
    sh(['uv', 'pip', 'install', '--python', f'{venv}/bin/python', '-r', req_file])
    open(stamp, 'w').write(want)

def ensure_tts():
    """Chatterbox venv + import check — only when the bundle has no ready speech (ячейка 6 вызывает её сама)."""
    ensure_venv('tts')
    sh([TTS_PY, '-c', 'import torch, perth, chatterbox.mtl_tts; assert perth.PerthImplicitWatermarker, "perth: нет pkg_resources (setuptools<81)"; print("tts ok: torch", torch.__version__, "cuda", torch.cuda.is_available())'])

# the animation venv is always needed; the TTS one is installed later only if the bundle has no speech_in.wav
ensure_venv('joy')

# quick import check: fails here, not in the middle of a long run
sh([JOY_PY, '-c', 'import torch, transformers, onnxruntime, cv2, librosa, pykalman, tyro; print("joy ok: torch", torch.__version__, "transformers", transformers.__version__, "cuda", torch.cuda.is_available())'])
print(f'установка: {time.time() - t0:.0f} с')

In [ ]:
#@title 3. Модели анимации (≈ 1.5 ГБ; повторно — мгновенно)
import os, time
os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')
from huggingface_hub import snapshot_download
t0 = time.time()
W = f'{JOY_DIR}/pretrained_weights'
# JoyVASA motion generator (hubert variant; the wav2vec2 one is "supported later" upstream)
snapshot_download('jdh-algo/JoyVASA', local_dir=f'{W}/JoyVASA', allow_patterns=['motion_generator/*', 'motion_template/*'])
# audio encoder: only the transformers weights, not the 1.1 GB fairseq checkpoint
snapshot_download('TencentGameMate/chinese-hubert-base', local_dir=f'{W}/chinese-hubert-base', allow_patterns=['config.json', 'preprocessor_config.json', 'pytorch_model.bin'])
alias = f'{W}/TencentGameMate:chinese-hubert-base'  # the path JoyVASA reads on Linux
if not os.path.exists(alias):
    os.symlink('chinese-hubert-base', alias)
# LivePortrait (human) + InsightFace buffalo_l; no animal models, no XPose (non-commercial)
snapshot_download('KwaiVGI/LivePortrait', local_dir=W, allow_patterns=['insightface/models/buffalo_l/*', 'liveportrait/base_models/*', 'liveportrait/retargeting_models/*', 'liveportrait/landmark.onnx'])
print(f'модели анимации готовы за {time.time() - t0:.0f} с')

def ensure_chatterbox_models():
    """Chatterbox Multilingual into the shared HF cache (from_pretrained finds it there) — only for TTS in Colab."""
    snapshot_download('ResembleAI/chatterbox', allow_patterns=['ve.pt', 't3_mtl23ls_v2.safetensors', 's3gen.pt', 'grapheme_mtl_merged_expanded_v1.json', 'conds.pt', 'Cangjie5_TC.json'])

## 4. Загрузка

Выберите `avatar_job.zip`. Можно вместе с ним выбрать и `vedit_avatar.py` (если обновляли скрипт) — иначе используется копия из архива.

Если кнопка выбора файла не появляется (блокировщик, сторонние cookies) — положите `avatar_job.zip` в корень Google Диска, выполните в новой ячейке `from google.colab import drive; drive.mount('/content/drive')` и задайте `UPLOAD = '/content/drive/MyDrive/avatar_job.zip'` перед этой ячейкой.

In [ ]:
#@title 4. Загрузка avatar_job.zip
import json, os, shutil, zipfile
WORK = '/content/avatar_work'
JOB = f'{WORK}/job'
OUT = f'{WORK}/out'
UPLOAD = globals().get('UPLOAD')  # optional: a path on a mounted Google Drive
if UPLOAD and os.path.exists(UPLOAD):
    bundle = UPLOAD
    sidecar_upload = None
else:
    from google.colab import files
    up = files.upload()
    zips = [n for n in up if n.lower().endswith('.zip')]
    assert zips, 'нужен avatar_job.zip (vedit avatar export или /avatar в боте)'
    bundle = zips[0]
    sidecar_upload = 'vedit_avatar.py' if 'vedit_avatar.py' in up else None
shutil.rmtree(JOB, ignore_errors=True)
os.makedirs(JOB)
with zipfile.ZipFile(bundle) as z:
    z.extractall(JOB)
if not os.path.exists(f'{JOB}/script.json'):
    sub = [d for d in os.listdir(JOB) if os.path.exists(f'{JOB}/{d}/script.json')]
    assert sub, 'в архиве нет script.json — это не avatar_job.zip'
    JOB = f'{JOB}/{sub[0]}'
if sidecar_upload:
    shutil.copy(sidecar_upload, f'{JOB}/vedit_avatar.py')
SIDECAR = f'{JOB}/vedit_avatar.py'
assert os.path.exists(SIDECAR), 'в архиве нет vedit_avatar.py — загрузите его вместе с архивом'
SCRIPT = json.load(open(f'{JOB}/script.json', encoding='utf-8'))
# a new bundle → a fresh output folder (the steps below also skip work whose inputs did not change)
if os.path.exists(f'{OUT}/job_id') and open(f'{OUT}/job_id').read() != SCRIPT['job']:
    shutil.rmtree(OUT)
os.makedirs(OUT, exist_ok=True)
open(f'{OUT}/job_id', 'w').write(SCRIPT['job'])
print('задача:', SCRIPT['job'], '| язык:', SCRIPT['language'], '| режим:', SCRIPT.get('mode', 'avatar'))
_sm = SCRIPT.get('speech_meta') or {}
print('голос:', f"готовая речь {_sm.get('engine')} / {_sm.get('voice')} (синтез пропускается)" if _sm else ('своя запись' if SCRIPT.get('speech') else ('клон по образцу' + (f" ({SCRIPT['voice_id']})" if SCRIPT.get('voice_id') else '') if SCRIPT.get('voice_ref') else 'стандартный Chatterbox')))
for i, line in enumerate(SCRIPT['lines'], 1):
    print(f'{i:2}. {line}')
if os.path.exists(f'{JOB}/portrait.png'):
    from IPython.display import Image, display
    display(Image(f'{JOB}/portrait.png', width=240))

In [ ]:
#@title 5. Настройки (можно не трогать)
MODE = 'auto'  #@param ['auto', 'avatar', 'voice_only']
EXAGGERATION = 0.5  #@param {type:'slider', min:0.25, max:1.0, step:0.05}
CFG_WEIGHT = 0.4  #@param {type:'slider', min:0.0, max:1.0, step:0.05}
CFG_SCALE = 2.0  #@param {type:'slider', min:1.0, max:4.0, step:0.5}
MAX_CHUNK_SECONDS = 20  #@param {type:'slider', min:8, max:40, step:2}
FPS = 25  #@param [25, 30] {type:'raw'}
HALF_PRECISION = False  #@param {type:'boolean'}
RUN_MODE = SCRIPT.get('mode', 'avatar') if MODE == 'auto' else MODE
print('режим:', RUN_MODE)
# EXAGGERATION — эмоциональность голоса (0.5 — нейтрально); CFG_WEIGHT — темп/следование образцу (0.3–0.5; с клоном
# голоса другого языка ставьте 0). CFG_SCALE — живость мимики и головы JoyVASA (2.0 — спокойно, 4.0 — активно).
# MAX_CHUNK_SECONDS — длина куска анимации (меньше — меньше памяти; куски режутся по паузам).
# FPS — 25 (родной) или 30. HALF_PRECISION — быстрее, но при чёрных квадратах на видео выключите.

In [ ]:
#@title 6. Речь
import os, sys
from IPython.display import Audio, display
if SCRIPT.get('speech'):
    # ready speech in the bundle (own recording or a Piper voice from vedit): no Chatterbox at all, only ffmpeg
    print('речь уже в архиве — Chatterbox не нужен')
    sh([sys.executable, SIDECAR, 'tts', '--job', JOB, '--out-dir', OUT])
else:
    ensure_tts()
    ensure_chatterbox_models()
    sh([TTS_PY, SIDECAR, 'tts', '--job', JOB, '--out-dir', OUT, '--device', 'cuda', '--exaggeration', str(EXAGGERATION), '--cfg-weight', str(CFG_WEIGHT)])
display(Audio(f'{OUT}/speech.wav'))

In [ ]:
#@title 7. Анимация лица
if RUN_MODE == 'voice_only':
    print('режим voice_only — анимация пропущена')
else:
    cmd = [JOY_PY, SIDECAR, 'animate', '--job', JOB, '--out-dir', OUT, '--device', 'cuda', '--joyvasa', JOY_DIR,
           '--cfg-scale', str(CFG_SCALE), '--max-chunk-seconds', str(MAX_CHUNK_SECONDS)]
    if HALF_PRECISION:
        cmd.append('--half')
    sh(cmd)

In [ ]:
#@title 8. Сборка и скачивание avatar_result.zip
import os, sys
from IPython.display import Video, display
sh([sys.executable, SIDECAR, 'finalize', '--job', JOB, '--out-dir', OUT, '--mode', RUN_MODE, '--fps', str(FPS)])
if RUN_MODE != 'voice_only':
    display(Video(f'{OUT}/avatar.mp4', embed=True, width=320))
from google.colab import files
files.download(f'{OUT}/avatar_result.zip')
print('дальше: vedit avatar import', SCRIPT['job'], 'avatar_result.zip  — или пришлите архив боту документом')

## Если что-то пошло не так

- **`GPU не подключён`** — Runtime → Change runtime type → T4 GPU. Бесплатный GPU иногда недоступен — попробуйте позже.
- **`CUDA out of memory`** в анимации — уменьшите `MAX_CHUNK_SECONDS` (например, 12) и перезапустите ячейку 7.
- **`JoyVASA не нашёл лицо`** — нужен фронтальный портрет: лицо крупно, глаза открыты, рот закрыт, без очков и сильного поворота.
- **Голос не похож** — образец 5–10 с чистой речи **на том же языке**, без музыки; попробуйте `CFG_WEIGHT` 0.3 и `EXAGGERATION` 0.5–0.7.
- **Пропущенные/искажённые слова** — перезапустите ячейку 6 (новый сид); числа и аббревиатуры пишите словами.
- **Сессия Colab отключилась** — Runtime → Run all: установка и модели в той же сессии не повторяются, при новой сессии — повторятся.
- Любая ошибка: последние строки вывода ячейки — пришлите их разработчику.